# Bays (2014) Figure 3 — Weighted Population Encoding
## Cued vs Uncued Items and Optimal Attentional Weighting

### Intellectual Foundation

Figure 2 encoded all items equally. Figure 3 introduces **differential attentional weighting**:
one item (the cued item) gets a higher encoding weight α_cued > 1, while others get α = 1.

This modifies encoding: log r^pre_i = Σ_k α_k · f_{i,k}(θ_k). Under DN, boosting one item's
weight necessarily *reduces* the effective gain for other items (zero-sum competition).

**Panels a:** Error distributions split by cued/uncued — cued items are more precise.
**Panel b:** Variance: cued < uncued, gap grows with set size.
**Panel c:** Kurtosis for cued and uncued.
**Panel d:** Optimal α* that minimises total expected error across cued and uncued probes.

### The Retro-Cue Effect
This is the mechanistic explanation of the "retro-cue benefit": when observers know which
item will be tested, they can reallocate gain to that item. The model predicts both the
BENEFIT (lower cued error) and the COST (higher uncued error) from the same mechanism.

### What to play with
- `ALPHA_RANGE`: How much weight can the cued item get
- `CUE_RATIO`: Probability ratio of probing cued vs uncued (3:1 in Bays)
- `SET_SIZES`: [2, 4, 8] — the cueing benefit grows with more items

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import logsumexp
import time

np.random.seed(42)

# --- PARAMETERS ---
M = 100; N_THETA = 64; N_TRIALS = 3000; N_TRIALS_SWEEP = 1000
T_D = 0.1; SIGMA_SQ = 1e-6; LAMBDA_BASE = 0.5; GAMMA = 100.0
SET_SIZES = [2, 4, 8]; CUE_RATIO = 3.0
ALPHA_RANGE = (1.0, 8.0); N_ALPHA = 10
SEED = 42; N_SEEDS = 2; N_BINS = 50
alpha_values = np.linspace(*ALPHA_RANGE, N_ALPHA)

In [ ]:
# ============================================================
# SELF-CONTAINED CORE: GP Population + DN + Poisson + ML Decoder
# ============================================================
# Replaces imports from core.encoder.*, core.decoder.*, bays_utils

def periodic_rbf_kernel(thetas, lengthscale):
    """Periodic squared-exponential kernel."""
    diff = thetas[:, None] - thetas[None, :]
    circ = np.abs(np.angle(np.exp(1j * diff)))
    return np.exp(-0.5 * (circ / lengthscale)**2)

def sample_gp_function(K, rng):
    """Sample one function from GP(0, K)."""
    L = np.linalg.cholesky(K + 1e-6 * np.eye(len(K)))
    return L @ rng.randn(len(K))

def generate_population(M, n_theta, lengthscale, n_locations=1, seed=42):
    """Generate M neurons with GP tuning at n_locations. Returns (thetas, f_all)."""
    rng = np.random.RandomState(seed)
    thetas = np.linspace(-np.pi, np.pi, n_theta, endpoint=False)
    K = periodic_rbf_kernel(thetas, lengthscale)
    f_all = []
    for loc in range(n_locations):
        f_loc = np.zeros((M, n_theta))
        for i in range(M):
            f_loc[i] = sample_gp_function(K, rng)
        f_all.append(f_loc)
    return thetas, f_all

def dn_pointwise(r_pre, gamma, sigma_sq):
    """DN: r^post_i = γ · r^pre_i / (σ² + mean(r^pre))."""
    D = sigma_sq + np.mean(r_pre)
    return gamma * r_pre / D

def generate_spikes(rates, T_d, rng):
    """Poisson spike counts."""
    return rng.poisson(np.maximum(rates, 0) * T_d)

def compute_log_likelihood(counts, g, T_d):
    """Full Poisson ML: LL(θ) = Σ_i [n_i·log g_i(θ) - g_i(θ)·T_d]."""
    log_g = np.log(np.maximum(g, 1e-30))
    return counts @ log_g - T_d * np.sum(g, axis=0)

def compute_circular_error(theta_true, theta_hat):
    """Signed circular error in [-π, π)."""
    return np.angle(np.exp(1j * (theta_hat - theta_true)))

def circular_variance(errors):
    """Circular variance: 1 - |mean(exp(iε))|."""
    return 1.0 - np.abs(np.mean(np.exp(1j * errors)))

def circular_kurtosis(errors):
    """Circular kurtosis: κ₂/V² where κ₂ = 1-|ρ₂|, V = circ var."""
    V = circular_variance(errors)
    rho2 = np.abs(np.mean(np.exp(2j * errors)))
    kappa2 = 1.0 - rho2
    return kappa2 / max(V**2, 1e-15) if V > 1e-10 else 0.0

def compute_deviation_from_normal(errors, n_bins=50):
    """Deviation of empirical dist from matched circular normal."""
    from scipy.stats import vonmises
    bin_edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    width = bin_edges[1] - bin_edges[0]
    emp, _ = np.histogram(errors, bins=bin_edges, density=True)
    V = circular_variance(errors)
    kappa_fit = max(0.01, 1.0 / V - 1) if V > 0.01 else 100.0
    vm_pdf = vonmises.pdf(centers, kappa_fit)
    return {'bin_centers': centers, 'empirical': emp, 'normal_fit': vm_pdf,
            'deviation': emp - vm_pdf}

# Factorised multi-location decoder
from scipy.special import logsumexp

def compute_spike_weighted_log_tuning(counts, f_list):
    """L_k(θ) = Σ_i n_i · f_{i,k}(θ) for each location k."""
    return [counts @ f_k for f_k in f_list]

def compute_marginal_log_likelihood_efficient(L_list, cued_idx):
    """Efficient factorised ML: L_c(θ) + Σ_{k≠c} logsumexp(L_k)."""
    ll = L_list[cued_idx].copy()
    for k in range(len(L_list)):
        if k != cued_idx:
            ll = ll + logsumexp(L_list[k])
    return ll

print("Core model loaded (GP + DN + Poisson + ML decoder)")

In [ ]:
# Weighted marginal log-likelihood (figure-3 specific)
def compute_weighted_marginal(L_list, alpha_weights, probed_idx):
    ll = alpha_weights[probed_idx] * L_list[probed_idx]
    for k in range(len(L_list)):
        if k != probed_idx:
            ll = ll + logsumexp(alpha_weights[k] * L_list[k])
    return ll

def run_weighted_trials(f_all, thetas, active_locs, cued_loc, probed_loc,
                        alpha_cued, gamma, T_d, sigma_sq, n_trials, rng):
    l = len(active_locs)
    f_active = [f_all[loc] for loc in active_locs]
    alpha_w = np.ones(l); alpha_w[cued_loc] = alpha_cued
    errors = np.empty(n_trials)
    for t in range(n_trials):
        theta_idx = rng.randint(N_THETA, size=l)
        log_rpre = np.zeros(M)
        for k in range(l):
            log_rpre += alpha_w[k] * f_active[k][:, theta_idx[k]]
        r_pre = np.exp(log_rpre - np.max(log_rpre))
        rates = dn_pointwise(r_pre, gamma, sigma_sq)
        counts = generate_spikes(rates, T_d, rng)
        L_list = compute_spike_weighted_log_tuning(counts, f_active)
        ll_m = compute_weighted_marginal(L_list, alpha_w, probed_loc)
        errors[t] = compute_circular_error(thetas[theta_idx[probed_loc]], thetas[np.argmax(ll_m)])
    return errors
print("Weighted trial engine ready.")

---
## Experiment: Optimal Weight Search + Cued vs Uncued Simulation

### What is occurring
**Phase 1:** For each set size, sweep α_cued and measure V_cued and V_uncued.
Find α* = argmin [p_cued · V_cued + (N-1) · p_uncued · V_uncued].

**Phase 2:** At α*, run full trials separately for cued and uncued probes.
Compute variance, kurtosis, and error distributions for each condition.

In [ ]:
# ============================================================
# OPTIMAL ALPHA SEARCH + CUED/UNCUED SIMULATION
# ============================================================
t0 = time.time()
max_locs = max(SET_SIZES)
thetas, f_all = generate_population(M, N_THETA, LAMBDA_BASE, max_locs, SEED)

# Phase 1: Find optimal alpha per set size
optimal_alphas = {}
for N in SET_SIZES:
    p_cued = CUE_RATIO / (CUE_RATIO + N - 1)
    p_uncued = 1.0 / (CUE_RATIO + N - 1)
    best_cost, best_alpha = np.inf, 1.0
    for alpha in alpha_values:
        rng_c = np.random.RandomState(SEED + int(alpha*100) + N)
        rng_u = np.random.RandomState(SEED + int(alpha*100) + N + 1000)
        ec = run_weighted_trials(f_all, thetas, tuple(range(N)), 0, 0,
                                 alpha, GAMMA, T_D, SIGMA_SQ, N_TRIALS_SWEEP, rng_c)
        eu = run_weighted_trials(f_all, thetas, tuple(range(N)), 0, min(1,N-1),
                                 alpha, GAMMA, T_D, SIGMA_SQ, N_TRIALS_SWEEP, rng_u)
        cost = p_cued * circular_variance(ec) + (N-1) * p_uncued * circular_variance(eu)
        if cost < best_cost:
            best_cost, best_alpha = cost, alpha
    optimal_alphas[N] = best_alpha
    print(f"  N={N}: α* = {best_alpha:.2f}")

# Phase 2: Full simulation at optimal alpha
results_fig3 = {}
for N in SET_SIZES:
    alpha = optimal_alphas[N]
    rng_c = np.random.RandomState(SEED + N*100)
    rng_u = np.random.RandomState(SEED + N*100 + 5000)
    ec = run_weighted_trials(f_all, thetas, tuple(range(N)), 0, 0,
                             alpha, GAMMA, T_D, SIGMA_SQ, N_TRIALS, rng_c)
    eu = run_weighted_trials(f_all, thetas, tuple(range(N)), 0, min(1,N-1),
                             alpha, GAMMA, T_D, SIGMA_SQ, N_TRIALS, rng_u)
    results_fig3[N] = {
        'alpha': alpha,
        'var_cued': circular_variance(ec), 'var_uncued': circular_variance(eu),
        'kurt_cued': circular_kurtosis(ec), 'kurt_uncued': circular_kurtosis(eu),
        'hist_cued': compute_deviation_from_normal(ec, N_BINS),
        'hist_uncued': compute_deviation_from_normal(eu, N_BINS),
    }
    print(f"  N={N}: V_cued={results_fig3[N]['var_cued']:.4f}, V_uncued={results_fig3[N]['var_uncued']:.4f}")
print(f"\nDone in {time.time()-t0:.1f}s")

In [ ]:
# ============================================================
# FOUR-PANEL FIGURE
# ============================================================
RED, GREEN, BLUE = '#CC2222', '#228B22', '#2255AA'
bins = results_fig3[SET_SIZES[0]]['hist_cued']['bin_centers']
n_ss = len(SET_SIZES)

fig = plt.figure(figsize=(16, 10))
outer = gridspec.GridSpec(1, 2, width_ratios=[2.5, 1], wspace=0.35)
gs_a = gridspec.GridSpecFromSubplotSpec(2, n_ss, subplot_spec=outer[0], hspace=0.35, wspace=0.3)
gs_bcd = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=outer[1], hspace=0.45)

# Panel a: error distributions (cued + uncued)
for row, (key, color, label) in enumerate([(('hist_cued', RED, 'cued')), (('hist_uncued', GREEN, 'uncued'))]):
    for col, N in enumerate(SET_SIZES):
        ax = fig.add_subplot(gs_a[row, col])
        emp = results_fig3[N][key]['empirical']
        ax.plot(bins, emp, color=color, lw=1.5)
        ax.set_xlim(-np.pi, np.pi); ax.set_title(f'{N}', fontsize=12, fontweight='bold')
        ax.set_xticks([-np.pi, 0, np.pi]); ax.set_xticklabels([r'$-\pi$', '0', r'$\pi$'])
        if col == 0:
            ax.set_ylabel(f'{label}\nprob density')

# Panel b: Variance
ax_b = fig.add_subplot(gs_bcd[0])
for key, color, label in [('var_cued', RED, 'cued'), ('var_uncued', GREEN, 'uncued')]:
    vals = [results_fig3[N][key] for N in SET_SIZES]
    ax_b.plot(SET_SIZES, vals, 'o-', color=color, lw=2, ms=6, label=label)
ax_b.set_xlabel('items'); ax_b.set_ylabel('variance'); ax_b.legend(fontsize=9)

# Panel c: Kurtosis
ax_c = fig.add_subplot(gs_bcd[1])
for key, color, label in [('kurt_cued', RED, 'cued'), ('kurt_uncued', GREEN, 'uncued')]:
    vals = [results_fig3[N][key] for N in SET_SIZES]
    ax_c.plot(SET_SIZES, vals, 'o-', color=color, lw=2, ms=6, label=label)
ax_c.set_xlabel('items'); ax_c.set_ylabel('kurtosis'); ax_c.legend(fontsize=9)

# Panel d: Optimal alpha
ax_d = fig.add_subplot(gs_bcd[2])
ax_d.bar(range(n_ss), [optimal_alphas[N] for N in SET_SIZES], 0.5,
         color=BLUE, alpha=0.85, edgecolor='black')
ax_d.axhline(1, color='gray', ls='--', lw=1)
ax_d.set_xticks(range(n_ss)); ax_d.set_xticklabels([str(n) for n in SET_SIZES])
ax_d.set_xlabel('items'); ax_d.set_ylabel(r'$\alpha_{cued}$')

fig.suptitle('GP Population Coding — Bays (2014) Fig 3: Cued vs Uncued', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()